# Week 1 Exercise – Business Insight Explainer

This notebook demonstrates a small tool that explains the business models of
technology companies using large language models.

It compares responses from:
- GPT-4o-mini (OpenAI)
- Llama 3.2 running locally with Ollama

The goal is to explore prompt design and compare the quality of responses
between hosted and local models.

In [ ]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv()

MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"
openrouter_url = "https://openrouter.ai/api/v1"

OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")

client = OpenAI(base_url=openrouter_url, api_key=os.getenv("OPENAI_KEY"))

In [ ]:
SYSTEM_PROMPT = """
You are an expert in startup and SaaS business models.

Provide clear and structured explanations that help developers
and entrepreneurs understand how technology companies operate.
"""

question = """
Analyze the business model of Stripe.

Provide:
1. Short summary
2. Revenue model
3. Target customers
4. Competitive advantage
"""

In [ ]:
def ask_gpt(question: str) -> str:
    """Query GPT model and return the response."""

    response = client.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
    )

    return response.choices[0].message.content

In [ ]:
gpt_result = ask_gpt(question)

display(Markdown("## GPT-4o-mini Response"))
display(Markdown(gpt_result))

In [ ]:
def ask_llama(question: str) -> str:
    """Query local Llama model via Ollama."""

    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": MODEL_LLAMA,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": question},
            ],
            "stream": False
        },
    )

    data = response.json()
    return data["message"]["content"]

In [ ]:
llama_result = ask_llama(question)

display(Markdown("## Llama 3.2 Response"))
display(Markdown(llama_result))

In [ ]:
print("Observation:")

print("""
GPT produced a more structured explanation and clearer breakdown of the business model.

Llama generated a shorter response but still captured the main revenue model
and core target users.

This shows how prompt structure can guide models to produce useful business insights.
""")